# 02. YOLO baseline and first submission
HW2: Multiple Object Video Detection (UCU AI 2026)

Needs a GPU. Input: the dataset produced by `01_data_eda_convert.ipynb` (added as a Kaggle Input) and the competition data (for test frames).
**Goal:** a valid submission that beats the leaderboard baseline.

## 0. Install and config

In [ ]:
!pip install -q ultralytics

In [ ]:
from pathlib import Path
import csv, time

from ultralytics import YOLO

DATA_DIR = Path("/kaggle/input/hw2-data/hw2_data")   # <-- SET ME: dataset saved from notebook 01
TEST_DIR = Path("/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026")  # <-- SET ME: folder with test frames
WORK = Path("/kaggle/working")

MODEL = "yolo11s.pt"     # use "yolov8s.pt" if this version is not available
EPOCHS = 15
IMGSZ = 640
BATCH = 16
SEED = 42
RUN_NAME = "yolo_baseline"

## 1. Dataset config
Input folders are read-only on Kaggle, so the yaml points to the input data but is written to the working folder.

In [ ]:
names = ["pedestrian", "people", "bicycle", "car", "van", "truck", "tricycle", "awning-tricycle", "bus", "motor"]
yaml_path = WORK / "data.yaml"
yaml_path.write_text(
    f"path: {DATA_DIR / 'yolo'}\ntrain: images/train\nval: images/val\nnames:\n"
    + "\n".join(f"  {i}: {n}" for i, n in enumerate(names)) + "\n")
print(yaml_path.read_text())

## 2. Train
Short run on purpose: the aim is a working pipeline, not the best score. Log epochs and settings into `experiments_log.md`.

In [ ]:
model = YOLO(MODEL)
model.train(data=str(yaml_path), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED,
            deterministic=True, project=str(WORK / "runs"), name=RUN_NAME, plots=True)

## 3. Validation metrics
These are the numbers for the report table (mAP, AP50, speed). AP_small comes later from the COCO evaluator, so that all models are compared with the same code.

In [ ]:
best = WORK / "runs" / RUN_NAME / "weights" / "best.pt"
model = YOLO(str(best))
m = model.val(data=str(yaml_path), imgsz=IMGSZ)
print(f"mAP50-95: {m.box.map:.4f} | mAP50: {m.box.map50:.4f}")
print({k: round(v, 1) for k, v in m.speed.items()}, "(ms per image)")

## 4. Submission
**Check the competition format first** (Data tab, `sample_submission.csv`) and adjust only `HEADER` and `rows_for_detection` below:
column order, `x y w h` vs `x1 y1 x2 y2`, class id from 0 or 1, what `image_id` looks like.

In [ ]:
# --- ADJUST TO THE COMPETITION FORMAT ---
HEADER = ["image_id", "class_id", "score", "x", "y", "w", "h"]

def rows_for_detection(image_id, cls, score, x1, y1, x2, y2):
    return [image_id, int(cls) + 1, round(float(score), 5),
            round(float(x1), 2), round(float(y1), 2), round(float(x2 - x1), 2), round(float(y2 - y1), 2)]
# -----------------------------------------

In [ ]:
test_files = sorted(p for p in TEST_DIR.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
print(len(test_files), "files found. First:", test_files[:2])   # sanity check: these must be TEST frames only

In [ ]:
t0 = time.time()
with open(WORK / "submission.csv", "w", newline="") as f:
    wr = csv.writer(f)
    wr.writerow(HEADER)
    for res in model.predict([str(p) for p in test_files], imgsz=IMGSZ, conf=0.001,
                             max_det=300, stream=True, verbose=False):
        image_id = Path(res.path).stem
        for b in res.boxes:
            x1, y1, x2, y2 = b.xyxy[0].tolist()
            wr.writerow(rows_for_detection(image_id, b.cls.item(), b.conf.item(), x1, y1, x2, y2))
print(f"done in {time.time() - t0:.0f}s -> {WORK / 'submission.csv'}")

## Next
1. Submit `submission.csv` on the competition page. If the score is not above the baseline, check the format and box scale first.
2. Record the result (val mAP and public score) in `experiments_log.md`, then commit the notebooks.